# Can Morocco's Minimum Wage Keep Up with Food Inflation?

## Objective

This project analyzes whether the purchasing power of Morocco's minimum wage has kept pace with food inflation between 2010 and 2025.

## Data Sources

- Haut-Commissariat au Plan (HCP)
- World Bank
- Government publications
- (Optional) FAOSTAT

## Author

Imane LAABAB

## Last Updated

July 2026

In [ ]:
!pip install requests beautifulsoup4 pandas python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 2.1 MB/s eta 0:00:00


In [ ]:
!pip install pdfplumber --quiet
import pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 667.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 35.0 MB/s eta 0:00:00


In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import time
import os
from docx import Document

BASE_LIST_URL = "https://www.hcp.ma/Actualite-Indices-des-prix-a-la-consommation-IPC_r349.html"
HEADERS = {"User-Agent": "Mozilla/5.0 (research project; contact: your_email@example.com)"}
DOCX_DIR = "hcp_docx_files"
os.makedirs(DOCX_DIR, exist_ok=True)

PAGE_STARTS = [None] + list(range(5, 245, 5))
# PAGE_STARTS = [None]

In [ ]:
# ---------- STEP 1: collect bulletin URLs from listing pages ----------

def get_page_url(start):
    if start is None:
        return BASE_LIST_URL
    return f"{BASE_LIST_URL}?start={start}&show=&order="


BULLETIN_PATTERN = re.compile(
    r"consommation.*?-du-mois-de-[a-zA-Zéèêûôâ]+-\d{4}", re.IGNORECASE
)

def scrape_listing_page(start):
    url = get_page_url(start)
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    bulletins = []
    BASE_DOMAIN = "https://www.hcp.ma"

    for h3 in soup.find_all("h3"):
        link_tag = h3.find("a")
        if not link_tag:
            continue
        href = link_tag.get("href")
        if href and BULLETIN_PATTERN.search(href):
            if not href.startswith("http"):
                href = BASE_DOMAIN + href
            bulletins.append({"title": link_tag.get_text(strip=True), "url": href})
    return bulletins


def collect_all_bulletin_urls():
    all_bulletins = []
    for i, start in enumerate(PAGE_STARTS):
        print(f"Listing page {i+1}/{len(PAGE_STARTS)}...")
        try:
            all_bulletins.extend(scrape_listing_page(start))
        except Exception as e:
            print(f"  Failed: {e}")
        time.sleep(1)

    df = pd.DataFrame(all_bulletins).drop_duplicates(subset=["url"])
    df.to_csv("hcp_bulletin_urls.csv", index=False)
    print(f"Collected {len(df)} bulletin URLs -> hcp_bulletin_urls.csv")
    return df

In [ ]:
# ---------- STEP 2: visit each bulletin, find the .docx attachment link ----------

# def extract_month_year(title):
#     match = re.search(r"mois\s+d[e']\s*([A-Za-zéûôâ]+)\s+(\d{4})", title, re.IGNORECASE)
#     return (match.group(1), int(match.group(2))) if match else (None, None)

def extract_month_year(title):
    match = re.search(r"mois\s+d[e'’]\s*([A-Za-zéûôâ]+)\s+(\d{4})", title, re.IGNORECASE)
    if match:
        return match.group(1), int(match.group(2))
    match = re.search(r"ann[eé]e\s+(\d{4})", title, re.IGNORECASE)
    if match:
        return "Décembre", int(match.group(1))
    return None, None

# def get_docx_link(bulletin_url):
#     resp = requests.get(bulletin_url, headers=HEADERS, timeout=15)
#     resp.raise_for_status()
#     soup = BeautifulSoup(resp.text, "html.parser")

#     docx_links = []
#     for a in soup.find_all("a", href=True):
#         href = a["href"]
#         if "/attachment/" in href:
#             img = a.find("img")
#             alt = img.get("alt", "").lower() if img else ""
#             docx_links.append((href, alt))

#     # Prefer French version (alt text contains "_fr")
#     for href, alt in docx_links:
#         if "_fr" in alt:
#             return href
#     return docx_links[0][0] if docx_links else None


def get_docx_link(bulletin_url):
    resp = requests.get(bulletin_url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    docx_links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/attachment/" not in href:
            continue

        # Check BOTH the image alt text AND the visible link text for language hints
        img = a.find("img")
        alt = img.get("alt", "").lower() if img else ""
        link_text = a.get_text(" ", strip=True).lower()

        combined_text = f"{alt} {link_text}"
        docx_links.append((href, combined_text))

    # Prefer French version — check multiple possible French indicators
    french_markers = ["_fr", "version française", "version francaise", "(fr)", "français"]
    for href, text in docx_links:
        if any(marker in text for marker in french_markers):
            return href

    # Fallback: avoid Arabic-marked links 
    arabic_markers = ["version arabe", "_ar", "(ar)"]
    non_arabic = [href for href, text in docx_links
                  if not any(marker in text for marker in arabic_markers)]
    if non_arabic:
        return non_arabic[0]

    return docx_links[0][0] if docx_links else None


def build_docx_index(bulletin_df):
    records = []
    for _, row in bulletin_df.iterrows():
        month, year = extract_month_year(row["title"])
        print(f"Fetching docx link for {row['title']}...")
        try:
            docx_url = get_docx_link(row["url"])
        except Exception as e:
            print(f"  Failed: {e}")
            docx_url = None
        records.append({
            "title": row["title"], "url": row["url"],
            "month": month, "year": year, "docx_url": docx_url
        })
        time.sleep(1)

    df = pd.DataFrame(records)
    df.to_csv("hcp_docx_index.csv", index=False)
    print(f"Saved docx index -> hcp_docx_index.csv")
    return df

In [ ]:
# # ---------- STEP 3: download each .docx ----------

# def is_valid_docx(path):
#     with open(path, "rb") as f:
#         return f.read(2) == b"PK"

# def is_valid_pdf(path):
#     with open(path, "rb") as f:
#         return f.read(4) == b"%PDF"

# def download_attachment(url, filename_base, retries=3):
#     """Downloads either docx or pdf, detects type from actual content, not URL."""
#     for attempt in range(retries):
#         try:
#             resp = requests.get(url, headers=HEADERS, timeout=20)
#             resp.raise_for_status()
#             content = resp.content
#             if content[:2] == b"PK":
#                 path = os.path.join(DOCX_DIR, f"{filename_base}.docx")
#                 filetype = "docx"
#             elif content[:4] == b"%PDF":
#                 path = os.path.join(DOCX_DIR, f"{filename_base}.pdf")
#                 filetype = "pdf"
#             else:
#                 if attempt == retries - 1:
#                     return None, None
#                 time.sleep(3)
#                 continue
#             with open(path, "wb") as f:
#                 f.write(content)
#             return path, filetype
#         except Exception:
#             if attempt == retries - 1:
#                 return None, None
#             time.sleep(3)
#     return None, None

In [ ]:
import re

def is_valid_docx(path):
    with open(path, "rb") as f:
        return f.read(2) == b"PK"

def is_valid_pdf(path):
    with open(path, "rb") as f:
        return f.read(4) == b"%PDF"

def is_valid_rtf(path):
    with open(path, "rb") as f:
        return f.read(5) == b"{\\rtf"

# Update download_attachment to also recognize RTF
def download_attachment(url, filename_base, retries=3):
    docx_path = os.path.join(DOCX_DIR, f"{filename_base}.docx")
    pdf_path = os.path.join(DOCX_DIR, f"{filename_base}.pdf")
    rtf_path = os.path.join(DOCX_DIR, f"{filename_base}.rtf")

    if os.path.exists(docx_path) and is_valid_docx(docx_path):
        return docx_path, "docx"
    if os.path.exists(pdf_path) and is_valid_pdf(pdf_path):
        return pdf_path, "pdf"
    if os.path.exists(rtf_path) and is_valid_rtf(rtf_path):
        return rtf_path, "rtf"

    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=20)
            resp.raise_for_status()
            content = resp.content
            if content[:2] == b"PK":
                path, filetype = docx_path, "docx"
            elif content[:4] == b"%PDF":
                path, filetype = pdf_path, "pdf"
            elif content[:5] == b"{\\rtf":
                path, filetype = rtf_path, "rtf"
            else:
                if attempt == retries - 1:
                    return None, None
                time.sleep(3)
                continue
            with open(path, "wb") as f:
                f.write(content)
            return path, filetype
        except Exception:
            if attempt == retries - 1:
                return None, None
            time.sleep(3)
    return None, None


def parse_rtf_tables(path):
    """Lightweight RTF table parser: splits on \\row and \\cell tokens."""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()

    # def clean_cell_text(cell):
    #     # Strip RTF control words/groups, keep visible text
    #     cell = re.sub(r"\{\\[^{}]*\}", "", cell)   # remove nested control groups
    #     cell = re.sub(r"\\[a-zA-Z]+-?\d*", "", cell)  # remove control words
    #     cell = cell.replace("{", "").replace("}", "")
    #     cell = re.sub(r"\\'e9", "é", cell)  # common accented-char escapes
    #     cell = re.sub(r"\\'e0", "à", cell)
    #     cell = re.sub(r"\\'e8", "è", cell)
    #     cell = re.sub(r"\\par", " ", cell)
    #     cell = re.sub(r"\s+", " ", cell).strip()
    #     return cell

    # rows_raw = raw.split(r"\row")
    # table_rows = []
    # for row in rows_raw:
    #     cells = row.split(r"\cell")
    #     cleaned = [clean_cell_text(c) for c in cells]
    #     cleaned = [c for c in cleaned if c]  # drop empties
    #     if cleaned:
    #         table_rows.append(cleaned)

    # # Group consecutive rows of similar length into "tables"
    # tables = []
    # current = []
    # last_len = None
    # for row in table_rows:
    #     if last_len is not None and abs(len(row) - last_len) > 2 and current:
    #         tables.append(pd.DataFrame(current))
    #         current = []
    #     current.append(row)
    #     last_len = len(row)
    # if current:
    #     tables.append(pd.DataFrame(current))

    # return tables

def clean_cell_text(cell):
    # Common accented-character escape sequences (RTF encodes these as \'XX hex codes)
    accent_map = {
        r"\'e9": "é", r"\'e8": "è", r"\'e0": "à", r"\'e7": "ç",
        r"\'ea": "ê", r"\'f4": "ô", r"\'fb": "û", r"\'e2": "â",
        r"\'ee": "î", r"\'f9": "ù", r"\'eb": "ë",
    }
    for code, char in accent_map.items():
        cell = cell.replace(code, char)

    # Remove any other hex-escape sequences we didn't map explicitly
    cell = re.sub(r"\\'[0-9a-fA-F]{2}", "", cell)

    # Remove control words IN PLACE (don't touch surrounding text)
    cell = re.sub(r"\\[a-zA-Z]+-?\d*", " ", cell)

    # Remove leftover braces and stray backslashes
    cell = cell.replace("{", " ").replace("}", " ").replace("\\", " ")

    # Collapse whitespace
    cell = re.sub(r"\s+", " ", cell).strip()
    return cell


def parse_any_tables(path, filetype):
    if filetype == "docx":
        return parse_docx_tables(path)
    elif filetype == "pdf":
        return parse_pdf_tables(path)
    elif filetype == "rtf":
        return parse_rtf_tables(path)
    return []

In [ ]:
# ---------- STEP 4: parse tables out of each .docx ----------

def parse_docx_tables(path):
    """Extract all tables from a .docx as a list of DataFrames."""
    doc = Document(path)
    tables = []
    for table in doc.tables:
        data = [[cell.text.strip() for cell in row.cells] for row in table.rows]
        if data:
            tables.append(pd.DataFrame(data[1:], columns=data[0]))
    return tables

def parse_pdf_tables(path):
    tables = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            for t in page.extract_tables():
                if t:
                    tables.append(pd.DataFrame(t))
    return tables

def parse_any_tables(path, filetype):
    if filetype == "docx":
        return parse_docx_tables(path)
    elif filetype == "pdf":
        return parse_pdf_tables(path)
    return []

def build_final_dataset(docx_index_df):
    all_tables = []
    for _, row in docx_index_df.iterrows():
        if not row.get("local_path") or not os.path.exists(str(row["local_path"])):
            continue
        try:
            tables = parse_docx_tables(row["local_path"])
            for t_idx, table in enumerate(tables):
                table["month"] = row["month"]
                table["year"] = row["year"]
                table["table_index"] = t_idx
                all_tables.append(table)
        except Exception as e:
            print(f"  Failed parsing {row['local_path']}: {e}")

    if all_tables:
        combined = pd.concat(all_tables, ignore_index=True)
        combined.to_csv("hcp_ipc_full_dataset.csv", index=False)
        print(f"Saved combined dataset -> hcp_ipc_full_dataset.csv ({len(combined)} rows)")
        return combined
    else:
        print("No tables extracted.")
        return pd.DataFrame()

In [ ]:
def classify_table_role(df):
    first_cell = str(df.iloc[0, 0]).strip().lower()
    if "divisions" in first_cell:
        entity = "category"
    elif "villes" in first_cell:
        entity = "city"
    else:
        return None, None, None

    new_header = df.iloc[0]
    df = df[1:].copy()
    raw_cols = list(new_header)
    seen = {}
    unique_cols = []
    for c in raw_cols:
        c = str(c).strip()
        seen[c] = seen.get(c, 0) + 1
        unique_cols.append(c if seen[c] == 1 else f"{c}_{seen[c]}")
    df.columns = unique_cols
    df = df.reset_index(drop=True)

    n_value_cols = df.shape[1] - 1  # excluding id column
    role = "monthly" if n_value_cols <= 4 else "yearly_combined"

    for i in range(1, df.shape[1]):
        col = df.columns[i]
        s = df.iloc[:, i].astype(str).str.replace(",", ".", regex=False)\
                                     .str.replace("−", "-", regex=False)\
                                     .str.replace("\n", " ", regex=False)
        df[col] = pd.to_numeric(s, errors="coerce")

    return entity, role, df

In [ ]:
def standardize_columns(df, id_col, role):
    """Rename by position so columns are consistent across years regardless of header text drift."""
    n = df.shape[1] - 1
    if role == "monthly":
        names = ["index_prev_period", "index_curr_period", "change_pct"][:n]
    else:  # yearly_combined
        names = ["index_ref_period", "index_curr_period", "change_pct",
                  "cum_index_prior", "cum_index_curr", "cum_change_pct"][:n]
    df = df.copy()
    df.columns = [id_col] + names
    return df

def process_document_full(local_path, filetype, month, year):
    tables = parse_any_tables(local_path, filetype)
    out = {"category_monthly": [], "category_yearly": [], "city": []}

    for t in tables:
        entity, role, cleaned = classify_table_role(t)
        if entity is None:
            continue
        id_col = "category" if entity == "category" else "city"
        std = standardize_columns(cleaned, id_col, role)
        std["month"] = month
        std["year"] = year

        if entity == "category" and role == "monthly":
            out["category_monthly"].append(std)
        elif entity == "category" and role == "yearly_combined":
            out["category_yearly"].append(std)
        elif entity == "city":
            # city table is usually the combined/wide one; keep whichever role it is
            out["city"].append(std)

    return out

In [ ]:
# RUN EVERYTHING
def run_full_pipeline_v2(docx_index_df):
    all_monthly, all_yearly, all_city = [], [], []

    for _, row in docx_index_df.iterrows():
        path, filetype = download_attachment(row["docx_url"], str(row["docx_url"]).rstrip("/").split("/")[-1])
        if not path:
            print(f"  Could not download: {row['title']}")
            continue
        print(f"Processing {row['title']} ({filetype})...")
        result = process_document_full(path, filetype, row["month"], row["year"])
        all_monthly.extend(result["category_monthly"])
        all_yearly.extend(result["category_yearly"])
        all_city.extend(result["city"])
        time.sleep(0.5)

    monthly_df = pd.concat(all_monthly, ignore_index=True) if all_monthly else pd.DataFrame()
    yearly_df = pd.concat(all_yearly, ignore_index=True) if all_yearly else pd.DataFrame()
    city_df = pd.concat(all_city, ignore_index=True) if all_city else pd.DataFrame()

    monthly_df.to_csv("hcp_monthly_category.csv", index=False)
    yearly_df.to_csv("hcp_yearly_category.csv", index=False)
    city_df.to_csv("hcp_city_indices.csv", index=False)

    print(f"\nhcp_monthly_category.csv: {len(monthly_df)} rows")
    print(f"hcp_yearly_category.csv: {len(yearly_df)} rows")
    print(f"hcp_city_indices.csv: {len(city_df)} rows")

    return monthly_df, yearly_df, city_df

In [ ]:
bulletin_df = collect_all_bulletin_urls()
build_docx_index(bulletin_df)

Listing page 1/49...
Listing page 2/49...
Listing page 3/49...
Listing page 4/49...
Listing page 5/49...
Listing page 6/49...
Listing page 7/49...
Listing page 8/49...
Listing page 9/49...
Listing page 10/49...
Listing page 11/49...
Listing page 12/49...
Listing page 13/49...
Listing page 14/49...
Listing page 15/49...
Listing page 16/49...
Listing page 17/49...
Listing page 18/49...
Listing page 19/49...
Listing page 20/49...
Listing page 21/49...
Listing page 22/49...
Listing page 23/49...
Listing page 24/49...
Listing page 25/49...
Listing page 26/49...
Listing page 27/49...
Listing page 28/49...
Listing page 29/49...
Listing page 30/49...
Listing page 31/49...
Listing page 32/49...
Listing page 33/49...
Listing page 34/49...
Listing page 35/49...
Listing page 36/49...
Listing page 37/49...
Listing page 38/49...
Listing page 39/49...
Listing page 40/49...
Listing page 41/49...
Listing page 42/49...
Listing page 43/49...
Listing page 44/49...
Listing page 45/49...
Listing page 46/49.

,title,url,month,year,docx_url
0,L'Indice des prix à la consommation (IPC) du m...,https://www.hcp.ma/L-Indice-des-prix-a-la-cons...,Juin,2026,https://www.hcp.ma/attachment/2893122/
1,L'Indice des prix à la consommation (IPC) du m...,https://www.hcp.ma/L-Indice-des-prix-a-la-cons...,Mai,2026,https://www.hcp.ma/attachment/2885946/
2,L'Indice des prix à la consommation (IPC) du m...,https://www.hcp.ma/L-Indice-des-prix-a-la-cons...,Mars,2026,https://www.hcp.ma/attachment/2868744/
3,L'Indice des prix à la consommation (IPC) du m...,https://www.hcp.ma/L-Indice-des-prix-a-la-cons...,Février,2026,https://www.hcp.ma/attachment/2857013/
4,L'Indice des prix à la consommation (IPC) du m...,https://www.hcp.ma/L-Indice-des-prix-a-la-cons...,Janvier,2026,https://www.hcp.ma/attachment/2847321/
...,...,...,...,...,...
132,L'Indice des prix à la consommation (IPC) du m...,https://www.hcp.ma/L-Indice-des-prix-a-la-cons...,mai,2010,https://www.hcp.ma/attachment/2234257/
133,L'Indice des prix à la consommation (IPC) du m...,https://www.hcp.ma/L-Indice-des-prix-a-la-cons...,mars,2010,https://www.hcp.ma/attachment/2234270/
134,L'Indice des prix à la consommation (IPC) du m...,https://www.hcp.ma/L-Indice-des-prix-a-la-cons...,février,2010,https://www.hcp.ma/attachment/2234277/
135,L’Indice des prix à la consommation (IPC) du m...,https://www.hcp.ma/L-Indice-des-prix-a-la-cons...,janvier,2010,https://www.hcp.ma/attachment/2234286/


In [ ]:
def retry_missing_docx_links(docx_index_df):
    """Re-attempt get_docx_link() only for rows where docx_url is still missing."""
    missing_mask = docx_index_df["docx_url"].isna()
    print(f"Retrying {missing_mask.sum()} rows with missing docx_url...")

    for idx in docx_index_df[missing_mask].index:
        url = docx_index_df.loc[idx, "url"]
        title = docx_index_df.loc[idx, "title"]
        try:
            docx_url = get_docx_link(url)
            docx_index_df.loc[idx, "docx_url"] = docx_url
            if docx_url:
                print(f"  Recovered: {title}")
            else:
                print(f"  Still no attachment found: {title}")
        except Exception as e:
            print(f"  Still failing: {title} -> {e}")
        time.sleep(1)

    docx_index_df.to_csv("hcp_docx_index.csv", index=False)
    return docx_index_df

In [ ]:
docx_index_df = pd.read_csv("hcp_docx_index.csv")
docx_index_df = retry_missing_docx_links(docx_index_df)

Retrying 1 rows with missing docx_url...
  Still no attachment found: Précision du HCP relative à l'Indice des prix à la consommation du mois de février 2018


In [ ]:
df = pd.read_csv("hcp_docx_index.csv")
nan_rows = df[df["month"].isna()]
print(f"{len(nan_rows)} rows with NaN month")
print(nan_rows["title"].tolist())

0 rows with NaN month
[]


In [ ]:
docx_index_df = pd.read_csv("hcp_docx_index.csv")
monthly_df, yearly_df, city_df = run_full_pipeline_v2(docx_index_df)

Processing L'Indice des prix à la consommation (IPC) du mois de Juin 2026 (docx)...
Processing L'Indice des prix à la consommation (IPC) du mois de Mai 2026 (docx)...
Processing L'Indice des prix à la consommation (IPC) du mois de Mars 2026 (docx)...
Processing L'Indice des prix à la consommation (IPC) du mois de Février 2026 (docx)...
Processing L'Indice des prix à la consommation (IPC) du mois de Janvier 2026 (docx)...
Processing L'Indice des prix à la consommation (IPC) du mois de Novembre 2025 (docx)...
Processing L'Indice des prix à la consommation (IPC) du mois de Septembre 2025 (docx)...
Processing L'Indice des prix à la consommation (IPC) du mois de Juillet 2025 (docx)...
Processing L'Indice des prix à la consommation (IPC) du mois de Juin 2025 (docx)...
Processing L'Indice des prix à la consommation (IPC) du mois de Mai 2025 (docx)...
Processing L'Indice des prix à la consommation (IPC) du mois de Mars 2025 (docx)...
Processing L'Indice des prix à la consommation (IPC) du mois

In [ ]:
# import shutil

# shutil.rmtree("/content/hcp_docx_files/")

In [ ]:
monthly_df = pd.read_csv("hcp_monthly_category.csv")
yearly_df = pd.read_csv("hcp_yearly_category.csv")
city_df = pd.read_csv("hcp_city_indices.csv")

print(f"Final dataset sizes:")
print(f"  hcp_monthly_category.csv: {len(monthly_df)} rows")
print(f"  hcp_yearly_category.csv: {len(yearly_df)} rows")
print(f"  hcp_city_indices.csv: {len(city_df)} rows")
print(f"\nYears covered: {sorted(monthly_df['year'].dropna().unique())}")

Final dataset sizes:
  hcp_monthly_category.csv: 1605 rows
  hcp_yearly_category.csv: 1783 rows
  hcp_city_indices.csv: 2260 rows

Years covered: [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


In [ ]:
hcp_year_cat = pd.read_csv('hcp_yearly_category.csv')
hcp_year_cat.head()
#

,category,index_ref_period,index_curr_period,change_pct,cum_index_prior,cum_index_curr,cum_change_pct,month,year
0,Produits alimentaires,131.0,128.0,-2.3,131.5,130.2,-1.0,Juin,2026.0
1,01 - Produits alimentaires et boissons non alc...,130.5,127.2,-2.5,131.0,129.4,-1.2,Juin,2026.0
2,02 - Boissons alcoolisées et tabac,144.9,150.5,3.9,144.5,149.9,3.7,Juin,2026.0
3,Produits non alimentaires,112.0,114.6,2.3,112.1,113.8,1.5,Juin,2026.0
4,03 - Articles d'habillements et chaussures,117.0,117.9,0.8,116.8,117.9,0.9,Juin,2026.0


In [ ]:
hcp_city_ipc = pd.read_csv('hcp_city_indices.csv')
hcp_city_ipc.head()

,city,index_ref_period,index_curr_period,change_pct,cum_index_prior,cum_index_curr,cum_change_pct,month,year
0,Agadir,119.8,119.4,-0.3,118.8,119.7,0.8,Juin,2026.0
1,Casablanca,119.2,118.9,-0.3,118.5,119.1,0.5,Juin,2026.0
2,Fès,122.3,121.3,-0.8,122.6,122.4,-0.2,Juin,2026.0
3,Kénitra,121.7,119.6,-1.7,120.9,120.8,-0.1,Juin,2026.0
4,Marrakech,121.1,121.1,0.0,121.3,121.2,-0.1,Juin,2026.0


In [ ]:
# 1. Coverage: how many months per year did we actually capture?
print(monthly_df.groupby("year")["month"].nunique().sort_index())

# 2. Expected category count per month (~15) and city count (~19) — flag outliers
print(monthly_df.groupby(["year","month"])["category"].nunique().describe())
print(city_df.groupby(["year","month"])["city"].nunique().describe())

# 3. Spot-check a known figure: Jan 2026 food index should read 119.0-ish (per HCP's own summary)
print(monthly_df[(monthly_df.year==2026) & (monthly_df.month=="Janvier") &
                  (monthly_df.category=="Produits alimentaires")])

# 4. NaN audit — how much numeric data failed to parse?
print(monthly_df.isna().sum())
print(yearly_df.isna().sum())
print(city_df.isna().sum())

# 5. Duplicate check
print(monthly_df.duplicated(subset=["category","month","year"]).sum())

year
2010.0     4
2011.0     3
2012.0     5
2013.0     9
2014.0    10
2015.0    10
2016.0     7
2022.0     8
2023.0     9
2024.0    10
2025.0     9
2026.0     5
Name: month, dtype: int64
count    89.000000
mean     13.370787
std       4.367707
min       0.000000
25%      15.000000
50%      15.000000
75%      15.000000
max      15.000000
Name: category, dtype: float64
count    98.000000
mean     16.826531
std       5.046231
min       2.000000
25%      18.000000
50%      18.000000
75%      19.000000
max      19.000000
Name: city, dtype: float64
Empty DataFrame
Columns: [category, index_prev_period, index_curr_period, change_pct, month, year]
Index: []
category             29
index_prev_period    24
index_curr_period    24
change_pct           24
month                75
year                 75
dtype: int64
category             15
index_ref_period     26
index_curr_period    26
change_pct           26
cum_index_prior      11
cum_index_curr       11
cum_change_pct       26
month            

In [ ]:
test_title = "L'Indice des prix à la consommation (IPC) du mois de Juin 2026"
print(extract_month_year(test_title))

('Juin', 2026)
